# VisionBridge — trained model check (Colab)

Inference/checking only. This notebook does not train the model or modify VisionBridge source code. It downloads the small ISL-CSLTR video dataset only to obtain one real sentence-level video, extracts the same 132-dim pose + 1404-dim face features used by VisionBridge, and runs the trained checkpoint through the real CTC decoder.

The training notebook is intentionally untouched.


## 1. Bootstrap repo + Python imports


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_ROOT = Path('/content/VisionBridge')
if not (REPO_ROOT / 'README.md').exists():
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO_ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'],check=True)
BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0,str(BACKEND_ROOT))
os.chdir(REPO_ROOT)
import app
print('Repo:',REPO_ROOT)
print('Python:',sys.version.split()[0])
print('APP IMPORT: PASS')


## 2. Check MediaPipe installation without forcing a reinstall

The current Colab failure was `AttributeError: module 'mediapipe' has no attribute 'solutions'`. Modern MediaPipe builds may not expose the legacy API from the top-level module. We therefore import the legacy Holistic module directly from `mediapipe.python.solutions`, which is the compatibility path for the existing VisionBridge extractor.


In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('mediapipe') is None:
    print('MediaPipe is missing; installing a Python-version-compatible 0.10.x release...')
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','mediapipe==0.10.35'],check=True)
else:
    print('MediaPipe already installed; no reinstall.')

import mediapipe as mp
print('MediaPipe version:', getattr(mp,'__version__','unknown'))

try:
    from mediapipe.python.solutions import holistic as mp_holistic
    print('Legacy Holistic import: PASS')
except Exception as exc:
    raise RuntimeError(
        'Could not import MediaPipe legacy Holistic. Do not use mp.solutions.holistic; '
        'this notebook requires `from mediapipe.python.solutions import holistic`. '
        f'Original error: {exc}'
    ) from exc


## 3. Locate the trained checkpoint + vocabulary


In [ ]:
import shutil

WEIGHTS = REPO_ROOT/'backend/app/models/weights/base_model.pt'
VOCAB = REPO_ROOT/'backend/app/models/weights/base_model.vocab.json'
if not WEIGHTS.exists() or not VOCAB.exists():
    from google.colab import files
    print('Upload BOTH base_model.pt and base_model.vocab.json')
    uploaded=files.upload()
    for name in ('base_model.pt','base_model.vocab.json'):
        if name not in uploaded:
            raise FileNotFoundError(f'Missing required artifact: {name}')
        shutil.copy(name,WEIGHTS.parent/name)
assert WEIGHTS.exists() and VOCAB.exists()
print('Weights:',WEIGHTS)
print('Vocab:',VOCAB)


## 4. Validate and load the trained model


In [ ]:
import torch
from app.training.isltranslate import SimpleCharTokenizer
from app.models.base_model import load_frozen_base_model, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH

tokenizer=SimpleCharTokenizer.load(VOCAB)
state=torch.load(WEIGHTS,map_location='cpu')
assert isinstance(state,dict) and 'output_head.weight' in state
checkpoint_vocab=int(state['output_head.weight'].shape[0])
assert checkpoint_vocab==tokenizer.vocab_size,f'Vocabulary mismatch: checkpoint={checkpoint_vocab}, tokenizer={tokenizer.vocab_size}'
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=load_frozen_base_model(str(WEIGHTS),vocab_size=tokenizer.vocab_size).to(device).eval()
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable==0,'Base model is not frozen.'
print('Device:',device)
print('Vocabulary:',tokenizer.vocab_size)
print('Trainable parameters:',trainable)
print('CHECKPOINT VALIDATION: PASS')


## 5. Download the small real ISL-CSLTR dataset (check-only notebook)

This is not the training dataset download. We only use the sentence-level video dataset already referenced by the project notebook so we can test one real sign-language clip. The full training pipeline remains untouched.


In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('kagglehub') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','kagglehub'],check=True)

import kagglehub, glob
dataset_path=kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset')
print('Dataset path:',dataset_path)

video_root_candidates=[]
for p in glob.glob(os.path.join(dataset_path,'**','*Sentence_Level*'),recursive=True):
    if os.path.isdir(p) and 'Video' in os.path.basename(p):
        video_root_candidates.append(p)
assert video_root_candidates, 'Could not find sentence-level video directory in downloaded dataset.'
VIDEO_ROOT=video_root_candidates[0]
video_files=[]
for ext in ('*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV'):
    video_files.extend(glob.glob(os.path.join(VIDEO_ROOT,'**',ext),recursive=True))
video_files=sorted(video_files)
assert video_files,'No sentence-level videos found.'
print('Video root:',VIDEO_ROOT)
print('Videos found:',len(video_files))


## 6. Select one real sentence video


In [ ]:
TEST_VIDEO=video_files[0]
from pathlib import Path
expected_text=Path(TEST_VIDEO).parent.name.replace('_',' ').strip()
print('Selected video:',TEST_VIDEO)
print('Expected label from dataset folder:',expected_text)


## 7. Extract real MediaPipe Holistic keypoints

IMPORTANT: use the repository extractor. Do not call `mp.solutions.holistic`; use the direct compatibility import from Step 2.


In [ ]:
from backend.scripts.extract_keypoints import extract_clip_keypoints

print('Extracting real MediaPipe keypoints from the selected video...')
with mp_holistic.Holistic(static_image_mode=False,model_complexity=1) as holistic:
    pose_np,face_np=extract_clip_keypoints(TEST_VIDEO,holistic)
print('Pose shape:',pose_np.shape)
print('Face shape:',face_np.shape)
assert pose_np.ndim==2 and pose_np.shape[1]==POSE_INPUT_DIM==132
assert face_np.ndim==2 and face_np.shape[1]==FACE_INPUT_DIM==1404
assert pose_np.shape[0]==face_np.shape[0] and pose_np.shape[0]>0
print('REAL KEYPOINT EXTRACTION: PASS')


## 8. Run the trained model on the real sequence


In [ ]:
from app.services.inference_service import decode_logits
from app.training.isltranslate import _downsample_to_max_length

pose_t=torch.from_numpy(pose_np).float()
face_t=torch.from_numpy(face_np).float()
pose_t,face_t=_downsample_to_max_length(pose_t,face_t,Path(TEST_VIDEO).stem)
assert pose_t.shape[0]<=MAX_SEQUENCE_LENGTH

with torch.inference_mode():
    logits=model(pose_t.unsqueeze(0).to(device),face_t.unsqueeze(0).to(device))
prediction,confidence=decode_logits(logits)

print('\n'+'='*60)
print('REAL VISIONBRIDGE PREDICTION')
print('='*60)
print('VIDEO:',Path(TEST_VIDEO).name)
print('GROUND TRUTH:',expected_text)
print('PREDICTED:',prediction)
print('CONFIDENCE:',round(float(confidence),4))
print('FRAMES USED:',pose_t.shape[0])
print('LOGITS:',tuple(logits.shape))
print('='*60)


## 9. Calculate CER


In [ ]:
def levenshtein(a,b):
    prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1):
            cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(0 if ca==cb else 1)))
        prev=cur
    return prev[-1]

truth=expected_text.lower().strip()
pred=prediction.lower().strip()
distance=levenshtein(pred,truth)
cer=distance/max(len(truth),1)
print('Truth:',truth)
print('Prediction:',pred)
print('Edit distance:',distance)
print('CER:',round(cer,4))
print('MODEL CHECK COMPLETE')
